# GeoLens + leafmap

[GeoLens](https://github.com/geolens-io/geolens) is a self-hosted spatial data
hub: a catalog, search, and open standards over data that stays on your own
infrastructure. It serves OGC API Features, OGC API Records (catalog search),
STAC 1.0, MVT vector tiles, and raster tiles baked by
[TiTiler](https://developmentseed.org/titiler/). Every one of them is plain
HTTP, so nothing here needs a GeoLens-specific client.

This notebook reads a live instance into [leafmap](https://leafmap.org/):
search the catalog, load a feature collection, push a filter to the server
with CQL2, and drop raster tiles on the map. `GEOLENS` below defaults to the
public demo, which needs no account and no key; point it at your own instance
by changing that one line.


In [ ]:
# Pinned so a leafmap or geopandas upgrade doesn't change what this notebook
# does out from under you. Safe to re-run: pip skips anything already at the
# pinned version.
%pip install -q leafmap==0.63.1 geopandas==1.1.4 requests==2.33.1


In [ ]:
import io

import geopandas as gpd
import leafmap.foliumap as leafmap
import requests

# Point this at your own instance to use your own catalog. The public demo
# answers every route below anonymously.
GEOLENS = "https://demo.getgeolens.com"
API = f"{GEOLENS}/api"


## 1. Discover datasets: search the catalog

`GET /api/search/datasets/` is OGC API Records underneath: it runs the phrase
against embeddings of each record's title, description and keywords, so a
hit doesn't need to share a word with the query. On the demo, this exact
phrase matches the meteorite landings dataset with none of its five words
in that dataset's title.


In [ ]:
resp = requests.get(f"{API}/search/datasets/", params={"q": "space rocks that fell to earth", "limit": 5}, timeout=30)
resp.raise_for_status()
results = resp.json()

print(f"{results['numberMatched']} match(es), meaning-matched rather than keyword-matched:")
for feature in results["features"]:
    props = feature["properties"]
    print(f"  {feature['id']}  {props['title']!r}  ({props['record_type']})")


## 2. Load a vector collection over OGC API Features

`/api/collections/{id}/items` returns plain GeoJSON, so geopandas can read it
directly. The two collections below are the demo's NYC subway layers, small
enough (496 stations, 29 service lines) that one `limit=2000` request holds
the whole thing; a layer with more rows would need to follow the `next` link
GeoLens paginates with, the same way
[`python/analyze.py`](../python/analyze.py) does.

One gotcha worth naming: `gpd.read_file(url)` looks tempting, but geopandas
issues its own pre-flight request with the bare `urllib` user agent to sniff
the format, and the demo's CDN answers that agent with 403 while answering
`requests` (and a browser, and GDAL/curl) with 200. Fetching the bytes
ourselves sidesteps the sniff.


In [ ]:
STATIONS = "724bf894-dc1a-418c-abc6-555798c44d7c"  # NYC Subway Stations (MTA)
LINES = "de602fbe-8b30-4755-924f-c9e7fd9613b6"      # NYC Subway Lines (MTA)


def read_collection(collection_id: str, **params) -> gpd.GeoDataFrame:
    '''Read one page of an OGC API - Features collection into a GeoDataFrame.'''
    resp = requests.get(
        f"{API}/collections/{collection_id}/items",
        params={"limit": 2000, **params},
        timeout=30,
    )
    resp.raise_for_status()
    return gpd.read_file(io.BytesIO(resp.content))


stations = read_collection(STATIONS)
lines = read_collection(LINES)

assert len(stations) > 400 and len(lines) > 20, "expected the full demo subway layers"
print(f"{len(stations)} stations, {len(lines)} service lines, both in {stations.crs}")


In [ ]:
m = leafmap.Map(center=[40.75, -73.98], zoom=11)
m.add_gdf(lines, layer_name="Subway lines", style={"color": "#4da3ff", "weight": 2})
m.add_gdf(stations, layer_name="Subway stations", style={"color": "#ffd166", "radius": 3, "fillOpacity": 0.9})
m


## 3. Filter server-side with CQL2

[OGC API Features Part 3](https://docs.ogc.org/is/19-079r2/19-079r2.html)
lets a client hand the server a filter instead of downloading everything and
filtering locally. GeoLens evaluates `filter=` server-side against the
`datasets` collection: the catalog itself, one record per dataset. That's
what the query below narrows, and it's a different question from filtering
the *rows inside* a dataset (the stations and lines above). This instance
answers CQL2 on the catalog today, and not yet on a dataset's own feature
collection, so the check below confirms the conformance class before relying
on it rather than assuming a specific version.


In [ ]:
conformance = requests.get(f"{API}/conformance", timeout=30).json()["conformsTo"]
if "http://www.opengis.net/spec/cql2/1.0/conf/basic-cql2" not in conformance:
    print("This instance doesn't advertise CQL2 support; skipping.")
else:
    cql2_filter = "title LIKE '%Subway%'"
    resp = requests.get(f"{API}/collections/datasets/items", params={"filter": cql2_filter}, timeout=30)
    resp.raise_for_status()
    matches = gpd.read_file(io.BytesIO(resp.content))

    print(f"CQL2 filter {cql2_filter!r} matched {len(matches)} of the catalog's ~30 datasets:")
    print(matches[["title"]].to_string(index=False))
    assert set(matches["title"]) == {"NYC Subway Lines (MTA)", "NYC Subway Stations (MTA)"}


## 4. Raster: COG tiles through TiTiler

GeoLens ingests a raster once and bakes it into XYZ tiles with TiTiler, so a
client never has to open the COG itself: point a tile layer at the template
and every viewer past that gets pixels, not a multi-hundred-megabyte file.
The demo publishes a DEM over the Matterhorn as one example. Outside the
DEM's footprint the route answers `204` by design (that's the server saying
there's no tile there, not a broken one), so the map below opens centered on
the peak.


In [ ]:
DEM = "6f03bafa-34b3-4902-9351-40ce09a8181f"  # swissALTI3D Matterhorn DEM (2m mosaic)
tile_url = f"{GEOLENS}/raster-tiles/{DEM}/tiles/{{z}}/{{x}}/{{y}}.png"

# Confirm the route is live before wiring it into the map.
probe = requests.get(f"{GEOLENS}/raster-tiles/{DEM}/tiles/12/2135/1457.png", timeout=30)
probe.raise_for_status()
assert probe.headers["content-type"] == "image/png"

m2 = leafmap.Map(center=[45.976, 7.658], zoom=12)
m2.add_tile_layer(url=tile_url, name="Matterhorn DEM", attribution="GeoLens demo")
m2


## 5. Optional: segment the DEM with samgeo

[`segment-geospatial`](https://samgeo.gishub.org/) runs Meta's Segment
Anything Model over a georeferenced raster. It needs `torch` and a
multi-hundred-megabyte model checkpoint, neither of which belongs in a
notebook that is supposed to run anywhere in a few seconds, so this cell is
off by default. Flip the flag and install the extra once, and it turns the
tile layer above into a local GeoTIFF and segments it.


In [ ]:
RUN_SEGMENTATION = False  # pip install segment-geospatial torch, then flip this on

if RUN_SEGMENTATION:
    from samgeo import SamGeo

    # samgeo segments a raster file, not a live tile source, so pull the tiles
    # over the DEM's footprint into a local GeoTIFF first.
    dem_bbox = [7.60, 45.95, 7.72, 46.01]
    leafmap.tms_to_geotiff("matterhorn.tif", dem_bbox, zoom=14, source=tile_url, to_cog=True)

    sam = SamGeo(model_type="vit_h", automatic=True)
    sam.generate("matterhorn.tif", output="matterhorn_segments.tif")
    m2.add_raster("matterhorn_segments.tif", layer_name="Segments", opacity=0.6)
    m2
else:
    print("Skipping: RUN_SEGMENTATION is False. See the README for what this needs.")


---

Read further: the [API reference](https://docs.getgeolens.com/guides/api/ogc/)
covers every route above, [`qgis/`](../qgis/) reads the same catalog from
desktop GIS, and [`python/analyze.py`](../python/analyze.py) does a
comparable spatial join in plain GeoPandas. If GeoLens is useful to you,
[star it on GitHub](https://github.com/geolens-io/geolens). That's how most
people find it.
